In [ ]:
import json
from pathlib import Path

# Keep the data folder in one variable so we can change it later.
DATA_DIR = Path("/content")

# Read a JSONL file: each nonempty line is one JSON example.
def load_jsonl(filename):
    with open(DATA_DIR / filename, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

# Load the training and validation examples into Python lists.
train_data = load_jsonl("sft_train.jsonl")
val_data = load_jsonl("sft_val.jsonl")

# Check how many examples were loaded.
print("Training rows:", len(train_data))
print("Validation rows:", len(val_data))

# Display the first training example with readable indentation.
print(json.dumps(train_data[0], indent=2))

In [ ]:
# Install the library used to load the tokenizer.
%pip install -q transformers

In [ ]:
import os

# Switch to an existing working folder.
os.chdir("/content")

print("Current folder:", os.getcwd())

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

# Load the tokenizer belonging to the assigned model.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Get the conversation from the first training example.
messages = train_data[0]["messages"]

# Show how the tokenizer formats this conversation for training.
formatted = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=False
)
print(formatted)

## Return a plain list of token IDs so len() counts tokens.
tokens = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=False,
    return_dict=False
)

print("\nToken count:", len(tokens))
print("Within 4096-token limit:", len(tokens) <= 4096)

In [ ]:
MAX_LENGTH = 4096

# Keep the original list and collect examples that fit.
train_filtered = []
dropped = 0

for row in train_data:
    # Count the full conversation, including the reference answer.
    token_ids = tokenizer.apply_chat_template(
        row["messages"],
        tokenize=True,
        add_generation_prompt=False,
        return_dict=False
    )

    if len(token_ids) <= MAX_LENGTH:
        train_filtered.append(row)
    else:
        dropped += 1

print("Training rows kept:", len(train_filtered))
print("Training rows dropped:", dropped)

In [ ]:
# Show which GPU is connected to this runtime.
!nvidia-smi

In [ ]:
# Install tools for loading the model onto the GPU in 4-bit.
%pip install -q accelerate bitsandbytes

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# Store model weights in 4-bit and perform calculations in fp16.
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

# Load the original model onto the GPU without training it.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map={"": 0},
    dtype=torch.float16
)

# Set the model to inference mode.
model.eval()

print("Base model loaded.")

In [ ]:
# Install experiment logging.
%pip install -q wandb

In [ ]:
import wandb

# Sign in using W&B's login prompt.
wandb.login(relogin=True)

In [ ]:
# Replace these with your team's W&B workspace and private project.
WANDB_ENTITY = "damians214-university-of-texas-at-dallas"
WANDB_PROJECT = "vibecheck baseline"

In [ ]:
# TRL runs supervised fine-tuning.
# PEFT adds LoRA adapters.
# Datasets prepares data for the trainer.
%pip install -q trl peft datasets

In [ ]:
import trl
import peft
import transformers

print("TRL:", trl.__version__)
print("PEFT:", peft.__version__)
print("Transformers:", transformers.__version__)

In [ ]:
from datasets import Dataset

# Separate each conversation into the input and the answer to learn.
training_rows = []

for row in train_filtered:
    messages = row["messages"]

    # Confirm the final message is the reference answer.
    assert messages[-1]["role"] == "assistant"

    training_rows.append({
        "prompt": messages[:-1],
        "completion": messages[-1:]
    })

train_dataset = Dataset.from_list(training_rows)

# Prepare validation examples in the same format as training.
validation_rows = []

for row in val_data:
    messages = row["messages"]
    assert messages[-1]["role"] == "assistant"

    validation_rows.append({
        "prompt": messages[:-1],
        "completion": messages[-1:]
    })

eval_dataset = Dataset.from_list(validation_rows)

print("Validation examples:", len(eval_dataset))

print("Training examples:", len(train_dataset))
print("Prompt roles:", [m["role"] for m in train_dataset[0]["prompt"]])
print("Completion roles:", [m["role"] for m in train_dataset[0]["completion"]])

In [ ]:
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)
from transformers import set_seed

set_seed(42)

# Prepare the 4-bit model for training with gradient checkpointing.
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True
)
model.config.use_cache = False

# Keep the assigned LoRA settings unchanged.
# Our only experiment change will be training for 2 epochs.
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    bias="none",
    task_type="CAUSAL_LM"
)

# Attach the trainable adapters to the frozen base model.
model = get_peft_model(model, lora_config)

# Show how many parameters will be trained.
model.print_trainable_parameters()

In [ ]:
import math
from trl import SFTConfig, SFTTrainer

# Batch size 1 × accumulation 8 gives an effective batch size of 8.
total_steps = math.ceil(len(train_dataset) / 8) * 2
warmup_steps = math.ceil(total_steps * 0.03)

training_config = SFTConfig(
    output_dir="/content/vibecheck-c-2epochs",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,
    max_length=4096,
    packing=False,
    completion_only_loss=True,
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    optim="adamw_torch",
    seed=42,
    data_seed=42,
    logging_steps=1,
    eval_strategy="epoch",
    per_device_eval_batch_size=1,
    report_to="wandb",
    run_name="qwen-r3-C-2epochs",
    save_strategy="epoch",
    save_total_limit=2,
    push_to_hub=False
)

# TRL applies the tokenizer's chat template to our structured messages.
trainer = SFTTrainer(
    model=model,
    args=training_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer
)

print("Trainer ready. Training has not started.")

In [ ]:
# Inspect one prepared example without starting training.
example = trainer.train_dataset[0]
batch = trainer.data_collator([example])

ids = batch["input_ids"][0]
labels = batch["labels"][0]

# -100 labels are ignored when calculating loss.
learned_ids = ids[labels != -100]

print("Tokens excluded from loss:", (labels == -100).sum().item())
print("Tokens included in loss:", len(learned_ids))
print("\nText the model will learn to produce:\n")
print(tokenizer.decode(learned_ids, skip_special_tokens=False))

In [ ]:
# Log training separately from the completed baseline run.
run = wandb.init(
    entity=WANDB_ENTITY,
    project=WANDB_PROJECT,
    name="qwen-r3-C-2epochs",
    save_code=False,
    config={
        "model": MODEL_NAME,
        "dataset": "sft_r3",
        "experiment": "C",
        "epochs": 2,
        "learning_rate": 2e-4,
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05,
        "target_modules": "all-linear",
        "max_length": 4096,
        "batch_size": 1,
        "gradient_accumulation": 8,
        "scheduler": "cosine",
        "warmup_steps": warmup_steps,
        "precision": "fp16",
        "quantization": "4-bit NF4",
        "completion_only_loss": True,
        "seed": 42,
        "training_rows": len(train_dataset),
        "dropped_rows": dropped
    }
)

In [ ]:
# Clean gradients and cast trainable parameters to float32 for fp16 training
trainer.model.zero_grad(set_to_none=True)
for parameter in trainer.model.parameters():
    if parameter.requires_grad:
        parameter.data = parameter.data.to(torch.float32)

# Train the LoRA adapters for two epochs.
training_result = trainer.train()

# Confirm training completed before saving.
completed_steps = trainer.state.global_step
planned_steps = trainer.state.max_steps

print("Completed optimizer steps:", completed_steps)
print("Planned optimizer steps:", planned_steps)

if completed_steps != planned_steps:
    raise RuntimeError("Training did not finish. Adapter was not saved.")

# Save the adapter and tokenizer in the checkpoints folder.
ADAPTER_DIR = "/content/training/checkpoints/vibecheck-c-adapter"

trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

run.finish()
print("Training complete. Adapter saved to:", ADAPTER_DIR)

In [ ]:
import shutil
from google.colab import files

# Package the saved adapter and tokenizer into a ZIP.
zip_path = shutil.make_archive(
    ADAPTER_DIR,
    "zip",
    ADAPTER_DIR
)

# Download the ZIP to your computer.
files.download(zip_path)